# Projekt - fáza č. 2

In [9]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.stats.api as sms
import scipy.stats as stats
from sklearn import preprocessing
from IPython.display import display
sns.set_theme(style="whitegrid")

## 2.1 A – Rozdelenie dát na trénovaciu a testovaciu množinu

Dáta delíme v pomere 80 % train a 20 % test. Od tohto kroku ďalej pracujeme iba s *train* množinou.

Každá jednotka (id_unit) má v dátach viac meraní (v priemere asi 7). Preto delíme podľa jednotiek, nie podľa riadkov - všetky merania jednej jednotky skončia buď v train, alebo v test. Keby sa tá istá jednotka dostala do oboch množín, model by si ju zapamätal a výsledky na teste by vyšli lepšie, než v skutočnosti sú.

Delenie je zároveň stratifikované na `fail`, takže podiel porúch (asi 12 %) je v oboch množinách rovnaký. `random_state` je pevne daný, aby bolo delenie vždy rovnaké.

In [10]:
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42   # pevny seed -> rovnake delenie pri kazdom spusteni
TEST_SIZE = 0.20    # pomer 80/20

df = pd.read_csv("./data/Ciste_data/merged_clean.csv")

# 5 foldov = kazdy fold je 20 % dat - jeden z nich pouzijeme ako test
sgkf = StratifiedGroupKFold(n_splits=round(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(df, df["fail"], groups=df["id_unit"]))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# kontrola: ziadna jednotka nie je v oboch mnozinach a nic sa nestratilo
assert set(train["id_unit"]).isdisjoint(test["id_unit"])
assert len(train) + len(test) == len(df)

for nazov, cast in [("cele data", df), ("train", train), ("test", test)]:
    print(nazov, "- riadky:", len(cast), "- jednotky:", cast["id_unit"].nunique(), "- podiel fail=1:", round(cast["fail"].mean(), 4))

train.to_csv("./data/Ciste_data/train_raw.csv", index=False, encoding="utf-8-sig")
test.to_csv("./data/Ciste_data/test_raw.csv", index=False, encoding="utf-8-sig")

cele data - riadky: 13772 - jednotky: 1999 - podiel fail=1: 0.1196
train - riadky: 11015 - jednotky: 1594 - podiel fail=1: 0.1197
test - riadky: 2757 - jednotky: 405 - podiel fail=1: 0.119
